# Bike Count Estimation — Group 21 Submission

Evaluates our best-performing **next-hour (t+1)** model on an unseen test dataset.

## How to run
1. Place the test file (`.xlsx`) next to this notebook (or update `TEST_DATA_PATH` in the *Configuration* cell).
2. Make sure `MODEL_GROUP21.pkl` sits next to this notebook.
3. Click **Run All**.
4. The final cell prints the **MSE** of our predictions against the ground truth.

The notebook re-creates the exact feature-engineering pipeline used at training time (lag/rolling features, cyclical time encoding, weather severity buckets, daylight features, NRW holiday/school/lecture flags), transforms the test data with the fitted preprocessor stored in the model bundle, predicts the next-hour bike count for every usable row, and reports the MSE on raw counts.

## 1. Configuration

**Edit the two paths below to point to the test file and the saved model.**

In [7]:
from pathlib import Path

# === EDIT THESE PATHS =========================================================
TEST_DATA_PATH = Path("challenge_hidden_test_dataset.xlsx")  # path to the test .xlsx file
MODEL_PATH     = Path("MODEL_GROUP21.pkl")                 # saved model bundle
# ==============================================================================

## 2. Imports

In [8]:
import re
import warnings
from datetime import date, timedelta

import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_squared_error

# `holidays` package supplies NRW school holiday dates for the feature engineering.
import holidays as _holidays_pkg

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
print("Imports OK.")

Imports OK.


## 3. Feature Engineering
- mixed `Weekday` string -> numeric weekday + text component
- cyclical sine/cosine encoding for `Hour`, `Day`, `Month`, weekday
- backward-looking lag features (1, 2, 3, 6, 12, 24, 48 hours) and rolling mean/std
- weather severity bucket + precipitation / snow flags
- closed-form sunrise/sunset features for Münster
- NRW public holidays + bridge days (auto-inferred year)
- WWU & FH Münster lecture periods + NRW school holidays
- temperature × humidity and wind × rain interactions

In [ ]:
# ---------- Weekday / cyclical / lag / rolling ----------
def parse_weekday_column(series: pd.Series) -> pd.DataFrame:
    weekday_num, weekday_text = [], []
    for val in series.astype(str):
        match = re.match(r"^(\d+)\s*(.*)", val.strip())
        if match:
            weekday_num.append(int(match.group(1)))
            text = match.group(2).strip()
            weekday_text.append(text if text else "Unknown")
        else:
            weekday_num.append(-1)
            weekday_text.append(val.strip())
    return pd.DataFrame({"weekday_num": weekday_num, "weekday_text": weekday_text})


def cyclical_encode(value: pd.Series, max_val: float):
    return np.sin(2 * np.pi * value / max_val), np.cos(2 * np.pi * value / max_val)


def add_lag_features(df, target_col="BikeCount", lags=None):
    if lags is None:
        lags = [1, 2, 3, 6, 12, 24, 48]
    for lag in lags:
        df[f"lag_{lag}"] = df[target_col].shift(lag)
    return df


def add_rolling_features(df, target_col="BikeCount", windows=None):
    if windows is None:
        windows = [3, 6, 12, 24]
    for w in windows:
        rolled = df[target_col].shift(1).rolling(window=w, min_periods=1)
        df[f"rolling_mean_{w}"] = rolled.mean()
        df[f"rolling_std_{w}"] = rolled.std().fillna(0)
    return df


# ---------- Weather severity ----------
WEATHER_SEVERITY = {
    "Sunny": 0, "Partly Cloudy": 0,
    "Cloudy": 1, "Overcast": 1, "Fog": 1, "Light Fog": 1, "Ice Fog": 1,
    "Drizzle": 2, "Occasional Drizzle": 2, "Light Rain": 2,
    "Occasional Light Rain": 2, "Light Shower": 2, "Occasional Rain": 2,
    "Light Snowfall": 2, "Occasional Light Snowfall": 2, "Light Snow Showers": 2,
    "Occasional Snowfall": 2, "Light Ice Rain": 2, "Occasional Ice Rain": 2,
    "Moderate Rainfall": 3, "Partially Moderate Rainfall": 3, "Moderate Snowfall": 3,
    "Occasional Moderate Snowfall": 3, "Occasional Drizzle with Thunderstorms": 3,
    "Heavy Rainfall": 4, "Partially Heavy Rainfall": 4, "Moderate to Heavy Shower": 4,
    "Moderate to Heavy Rain with Thunderstorms": 4, "Heavy Snowfall": 4,
    "Moderate to Heavy Snowfall": 4, "Moderate to Heavy Snowfall with Thunderstorms": 4,
    "Snowstorm": 4, "Snowdrifts": 4, "Occasional Thunderstorms and Precipitation": 4,
}
_PRECIP_KEYWORDS = ("rain", "drizzle", "shower", "snow", "thunderstorm", "ice")
_SNOW_KEYWORDS = ("snow", "snowstorm", "snowdrift")


def add_weather_features(df):
    df = df.copy()
    weather = df["Weather"].astype(str)
    is_null = weather.str.contains("Null", case=False, na=False)
    weather = weather.where(~is_null, other="Cloudy")
    df["weather_severity"] = weather.map(lambda l: WEATHER_SEVERITY.get(l, 1)).astype(np.int64)
    df["is_precipitation"] = weather.map(
        lambda l: int(any(k in l.lower() for k in _PRECIP_KEYWORDS))
    ).astype(np.int64)
    df["is_snow"] = weather.map(
        lambda l: int(any(k in l.lower() for k in _SNOW_KEYWORDS))
    ).astype(np.int64)
    return df


# ---------- Sunrise / sunset (Münster) ----------
LAT_DEG = 51.96
LON_DEG = 7.63
TZ_OFFSET_HOURS = 1.0


def _solar_event_hours(month, day, year=2000):
    # Use a leap year (2000) so Feb 29 is representable.
    try:
        doy = (date(year, month, day) - date(year, 1, 1)).days + 1
    except ValueError:
        doy = (date(year, month, day - 1) - date(year, 1, 1)).days + 1
    gamma = 2 * np.pi / 365.0 * (doy - 1 + 0.5)
    eq_time = 229.18 * (0.000075 + 0.001868*np.cos(gamma) - 0.032077*np.sin(gamma)
                        - 0.014615*np.cos(2*gamma) - 0.040849*np.sin(2*gamma))
    decl = (0.006918 - 0.399912*np.cos(gamma) + 0.070257*np.sin(gamma)
            - 0.006758*np.cos(2*gamma) + 0.000907*np.sin(2*gamma)
            - 0.002697*np.cos(3*gamma) + 0.00148*np.sin(3*gamma))
    lat_rad = np.radians(LAT_DEG)
    cos_ha = (np.cos(np.radians(90.833)) - np.sin(lat_rad)*np.sin(decl)) \
             / (np.cos(lat_rad)*np.cos(decl))
    cos_ha = np.clip(cos_ha, -1.0, 1.0)
    ha_deg = np.degrees(np.arccos(cos_ha))
    noon_min = 720.0 - 4.0*LON_DEG - eq_time + 60.0*TZ_OFFSET_HOURS
    return (noon_min - 4.0*ha_deg) / 60.0, (noon_min + 4.0*ha_deg) / 60.0


def add_daylight_features(df):
    df = df.copy()
    unique_dates = df[["Month", "Day"]].drop_duplicates()
    solar = {(int(r["Month"]), int(r["Day"])): _solar_event_hours(int(r["Month"]), int(r["Day"]))
             for _, r in unique_dates.iterrows()}
    sunrise = df.apply(lambda r: solar[(int(r["Month"]), int(r["Day"]))][0], axis=1)
    sunset = df.apply(lambda r: solar[(int(r["Month"]), int(r["Day"]))][1], axis=1)
    hour_f = df["Hour"].astype(float)
    df["hours_since_sunrise"] = hour_f - sunrise
    df["hours_until_sunset"] = sunset - hour_f
    return df


# ---------- NRW public holidays ----------
def _easter_sunday(year):
    a = year % 19; b = year // 100; c = year % 100
    d = b // 4; e = b % 4; f = (b + 8) // 25; g = (b - f + 1) // 3
    h = (19*a + b - d - g + 15) % 30; i = c // 4; k = c % 4
    l = (32 + 2*e + 2*i - h - k) % 7; m = (a + 11*h + 22*l) // 451
    month = (h + l - 7*m + 114) // 31
    day = ((h + l - 7*m + 114) % 31) + 1
    return date(year, month, day)


def nrw_holidays(year):
    easter = _easter_sunday(year)
    return {
        date(year, 1, 1): "Neujahr", date(year, 5, 1): "Tag der Arbeit",
        date(year, 10, 3): "Tag der Deutschen Einheit",
        date(year, 11, 1): "Allerheiligen",
        date(year, 12, 25): "1. Weihnachtstag", date(year, 12, 26): "2. Weihnachtstag",
        easter + timedelta(days=-2): "Karfreitag",
        easter + timedelta(days=1): "Ostermontag",
        easter + timedelta(days=39): "Christi Himmelfahrt",
        easter + timedelta(days=50): "Pfingstmontag",
        easter + timedelta(days=60): "Fronleichnam",
    }


def infer_year(df, candidate_range=(2010, 2030)):
    sample = df[["Month", "Day", "Weekday"]].drop_duplicates(subset=["Month", "Day"])
    matches = []
    for yr in range(candidate_range[0], candidate_range[1] + 1):
        ok = True
        for _, row in sample.iterrows():
            try:
                d = date(yr, int(row["Month"]), int(row["Day"]))
            except ValueError:
                ok = False; break
            if d.weekday() != int(row["Weekday"]):
                ok = False; break
        if ok:
            matches.append(yr)
    if not matches:
        raise ValueError(f"No year in {candidate_range} matches the calendar.")
    if len(matches) > 1:
        warnings.warn(f"Multiple candidate years: {matches}. Using {matches[-1]}.")
    return matches[-1]


def add_holiday_features(df, year):
    df = df.copy()
    holidays = nrw_holidays(year)
    holiday_dates = set(holidays.keys())
    jan1 = date(year, 1, 1)
    days_in_year = 366 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 365
    all_days = [jan1 + timedelta(days=i) for i in range(days_in_year)]
    holiday_flag = {d: (d in holiday_dates) for d in all_days}
    bridge_flag = {}
    for d in all_days:
        wd = d.weekday()
        if holiday_flag[d] or wd >= 5:
            bridge_flag[d] = False; continue
        prev_day = d - timedelta(days=1); next_day = d + timedelta(days=1)
        if wd == 4 and holiday_flag.get(prev_day, False):
            bridge_flag[d] = True
        elif wd == 0 and holiday_flag.get(next_day, False):
            bridge_flag[d] = True
        else:
            bridge_flag[d] = False
    row_dates = [date(year, int(m), int(d)) for m, d in zip(df["Month"], df["Day"])]
    df["is_holiday"] = np.array([1 if holiday_flag[d] else 0 for d in row_dates], dtype=np.int64)
    df["is_bridge_day"] = np.array([1 if bridge_flag[d] else 0 for d in row_dates], dtype=np.int64)
    return df


# ---------- WWU/FH lecture periods + NRW school holidays ----------
def _date_in_ranges(d, ranges):
    return any(start <= d <= end for start, end in ranges)


def _first_weekday_on_or_after(start, weekday):
    return start + timedelta(days=(weekday - start.weekday()) % 7)


def wwu_lecture_ranges(year):
    easter = _easter_sunday(year)
    ranges = []
    wise_prev_end = _first_weekday_on_or_after(date(year, 2, 1), 4)
    ranges.append((date(year, 1, 7), wise_prev_end))
    sose_start = _first_weekday_on_or_after(date(year, 4, 1), 0)
    if sose_start == easter + timedelta(days=1):
        sose_start += timedelta(days=7)
    sose_end = sose_start + timedelta(weeks=14, days=4)
    pf_tue = easter + timedelta(days=51)
    pf_fri = easter + timedelta(days=54)
    if sose_start <= pf_tue <= sose_end:
        ranges.append((sose_start, pf_tue - timedelta(days=1)))
        ranges.append((pf_fri + timedelta(days=1), sose_end))
    else:
        ranges.append((sose_start, sose_end))
    wise_start = _first_weekday_on_or_after(date(year, 10, 7), 0)
    ranges.append((wise_start, date(year, 12, 22)))
    return ranges


def fh_muenster_lecture_ranges(year):
    ranges = []
    wise_prev_end = _first_weekday_on_or_after(date(year, 2, 5), 4)
    ranges.append((date(year, 1, 7), wise_prev_end))
    sose_start = _first_weekday_on_or_after(date(year, 3, 15), 0)
    sose_end = sose_start + timedelta(weeks=16, days=4)
    ranges.append((sose_start, sose_end))
    wise_start = _first_weekday_on_or_after(date(year, 9, 22), 0)
    ranges.append((wise_start, date(year, 12, 22)))
    return ranges


def _consecutive_runs(dates):
    if not dates:
        return []
    dates = sorted(set(dates))
    ranges = []
    run_start = prev = dates[0]
    for d in dates[1:]:
        if (d - prev).days == 1:
            prev = d; continue
        ranges.append((run_start, prev)); run_start = prev = d
    ranges.append((run_start, prev))
    return ranges


def nrw_school_holiday_ranges(year):
    de_nw_school = _holidays_pkg.country_holidays(
        "DE", subdiv="NW", categories=("school",), years=year
    )
    return _consecutive_runs(list(de_nw_school.keys()))


def add_semester_and_school_features(df, year):
    df = df.copy()
    wwu_ranges = wwu_lecture_ranges(year)
    fh_ranges = fh_muenster_lecture_ranges(year)
    school_ranges = nrw_school_holiday_ranges(year)
    row_dates = [date(year, int(m), int(d)) for m, d in zip(df["Month"], df["Day"])]
    df["is_wwu_lecture"] = np.array(
        [1 if _date_in_ranges(d, wwu_ranges) else 0 for d in row_dates], dtype=np.int64)
    df["is_fh_lecture"] = np.array(
        [1 if _date_in_ranges(d, fh_ranges) else 0 for d in row_dates], dtype=np.int64)
    df["is_school_holiday"] = np.array(
        [1 if _date_in_ranges(d, school_ranges) else 0 for d in row_dates], dtype=np.int64)
    return df


# ---------- Master pipeline ----------
def build_features(df, is_training=True, target_col="BikeCount"):
    df = df.copy()
    weekday_parsed = parse_weekday_column(df["Weekday"])
    df["weekday_num"] = weekday_parsed["weekday_num"]
    df["weekday_text"] = weekday_parsed["weekday_text"]
    df.drop(columns=["Weekday"], inplace=True)

    df_for_year = df[["Month", "Day"]].copy()
    df_for_year["Weekday"] = df["weekday_num"]
    year = infer_year(df_for_year)
    df = add_holiday_features(df, year=year)
    df = add_semester_and_school_features(df, year=year)
    df = add_weather_features(df)
    df = add_daylight_features(df)

    df["hour_sin"], df["hour_cos"] = cyclical_encode(df["Hour"], 24)
    df["day_sin"], df["day_cos"] = cyclical_encode(df["Day"], 31)
    df["month_sin"], df["month_cos"] = cyclical_encode(df["Month"], 12)
    df["weekday_sin"], df["weekday_cos"] = cyclical_encode(df["weekday_num"], 7)

    df["is_weekend"] = (df["weekday_num"] >= 5).astype(int)
    df["is_rush_hour"] = df["Hour"].isin([7, 8, 9, 16, 17, 18]).astype(int)
    df["is_night"] = df["Hour"].isin(list(range(0, 6))).astype(int)

    if target_col in df.columns:
        df = add_lag_features(df, target_col)
        df = add_rolling_features(df, target_col)

    df["temp_humidity"] = df["Temperature (°C)"] * df["Humidity (%)"]
    df["wind_rain"] = df["Wind (km/h)"] * df["Rain (mm)"]
    return df


print("Feature engineering functions defined.")

## 4. Load Test Data

In [10]:
assert TEST_DATA_PATH.exists(), f"Test file not found: {TEST_DATA_PATH.resolve()}"

if TEST_DATA_PATH.suffix.lower() in (".xlsx", ".xls"):
    df_test_raw = pd.read_excel(TEST_DATA_PATH)
else:
    df_test_raw = pd.read_csv(TEST_DATA_PATH)

print(f"Loaded {TEST_DATA_PATH.name}: shape = {df_test_raw.shape}")
print(f"Columns: {df_test_raw.columns.tolist()}")
assert "BikeCount" in df_test_raw.columns, \
    "Test file must contain a 'BikeCount' column as ground truth."
df_test_raw.head()

Loaded challenge_hidden_test_dataset.xlsx: shape = (8783, 10)
Columns: ['Month', 'Day', 'Hour', 'Weekday', 'Weather', 'Temperature (°C)', 'Humidity (%)', 'Rain (mm)', 'Wind (km/h)', 'BikeCount']


,Month,Day,Hour,Weekday,Weather,Temperature (°C),Humidity (%),Rain (mm),Wind (km/h),BikeCount
0,1,1,0,0,Occasional Rain,7,76,0.0,32,84
1,1,1,1,0,Occasional Rain,7,72,0.0,31,156
2,1,1,2,0,Occasional Rain,7,72,0.0,33,203
3,1,1,3,0,Occasional Rain,7,73,0.0,31,267
4,1,1,4,0,Overcast,7,70,0.0,32,147


## 5. Load Trained Model

In [11]:
assert MODEL_PATH.exists(), f"Model file not found: {MODEL_PATH.resolve()}"

bundle = joblib.load(MODEL_PATH)
MODEL_TYPE   = bundle["model_type"]
preprocessor = bundle["preprocessor"]
NUM_COLS     = bundle["num_cols"]
CAT_COLS     = bundle["cat_cols"]
model        = bundle["model"]

print(f"Loaded MODEL_GROUP21.pkl  |  type = {MODEL_TYPE}")
if "h1_mse_train" in bundle:
    print(f"Training-set H1 validation MSE: {bundle['h1_mse_train']:.2f}")
print(f"Estimator: {type(model).__name__}")

Loaded MODEL_GROUP21.pkl  |  type = xgb
Training-set H1 validation MSE: 5277.03
Estimator: XGBRegressor


## 6. Apply Feature Engineering to Test Data

In [13]:
df_feat = build_features(df_test_raw, is_training=True)

# Build next-hour target (t+1).
df_feat["target_h1"] = df_feat["BikeCount"].shift(-1)

# Drop rows where any required column is NaN. This removes the first
# 48 rows (lag boundary), the very last row (no t+1 target), and any row with
# missing weather/target values in the raw data.
eval_lag_cols = [c for c in df_feat.columns if c.startswith("lag_") or c.startswith("rolling_")]
dropna_cols = NUM_COLS + CAT_COLS + eval_lag_cols + ["target_h1"]
dropna_cols = [c for c in dropna_cols if c in df_feat.columns]

df_eval = df_feat.dropna(subset=dropna_cols).reset_index(drop=True)
print(f"Engineered shape:  {df_feat.shape}")
print(f"Usable rows after NaN handling: {len(df_eval)}")

TypeError: 'NoneType' object is not subscriptable

## 7. Predict & Inspect

In [ ]:
X_eval = preprocessor.transform(df_eval)
y_true = df_eval["target_h1"].values.astype(float)
predictions = np.clip(model.predict(X_eval), 0.0, None)

mse = float(mean_squared_error(y_true, predictions))
rmse = float(np.sqrt(mse))
mae = float(np.mean(np.abs(predictions - y_true)))

print(f"Samples scored: {len(y_true)}")
print(f"RMSE: {rmse:.4f}    MAE: {mae:.4f}")

# Preview predicted vs. actual
preview = pd.DataFrame({
    "predicted_BikeCount": np.round(predictions, 2),
    "actual_BikeCount":    y_true,
})
preview["abs_error"] = (preview["predicted_BikeCount"] - preview["actual_BikeCount"]).abs()
preview.head(20)

Samples scored: 8702
RMSE: 33.3837    MAE: 14.3616


,predicted_BikeCount,actual_BikeCount,abs_error
0,13.330000,13.0,0.330000
1,10.260000,11.0,0.740000
2,7.570000,7.0,0.570000
3,14.350000,12.0,2.350000
4,36.020000,43.0,6.980000
5,151.699997,156.0,4.300003
6,389.750000,384.0,5.750000
7,416.450012,405.0,11.450012
8,355.839996,346.0,9.839996
9,293.570007,300.0,6.429993


## 8. Final MSE

In [ ]:
print(f"MSE (next-hour bike count prediction): {mse:.10f}")

MSE (next-hour bike count prediction): 1114.4710157757
